In [ ]:
import matplotlib.pyplot as mpl
import numpy as np
import pandas as pd
import re

x_keys = list()
full_df = pd.read_csv("full_results.csv")
# percentages = full_df.sort_values(by=["Percentage"])["Percentage"].unique() # If we want to use all percentages
percentages = [10,30,50,70,90]
full_df.loc[full_df["technique"].str.contains("OG"), ["Percentage"]] = full_df[full_df["technique"].str.contains("OG")]["Percentage"].apply(lambda p: p - 10 if p not in percentages else p)
project_features = pd.read_csv("program_features.csv")
project_features["MS"] = project_features["Killed"] / project_features["Total_mutants"]
projects = list(project_features.sort_values(by=["MS"])["project"].unique())
projects_map = {"Antomology":"antomology", "Javancss":"javancss", "Commons-cli":"cli", "Commons-csv":"csv", "Commons-dbutils":"dbutils", "Jettison":"jettison", "Commons-validator":"validator", "Commons-codec":"codec", "Commons-text":"text", "Jaxen":"jaxen", "Javautil":"javautil", "Jodatime":"jodatime","Commons-net":"net","Commons-lang3":"lang3","Jgrapht-core":"jgrapht"}
projects = list(map(lambda p: projects_map.get(p, p), projects))
# techniques = full_df.sort_values(by=["technique"])["technique"].unique()
techniques = ["random", "SL", "SC", "SO", "HG_1", "HG_2", "HG_3", "OG_1", "OG_2", "OG_3"]


fig, axs = mpl.subplots(len(techniques), len(percentages), figsize=(50, 50))
for p_index, percentage in enumerate(percentages):
    for t_index, technique in enumerate(techniques):
        values = full_df[(full_df["Percentage"] == percentage) 
                                  & (full_df["technique"] == technique)].sort_values(by="project")
        for project in projects:
            if project in values["project"].values:
                axs[t_index, p_index].bar(projects.index(project) + 1, 
                                          values[values["project"] == project]["median_error"],
                                          yerr= values[values["project"] == project]["std_error"]/2,
                                          color = mpl.colormaps["tab20"](projects.index(project)) #See https://matplotlib.org/stable/users/explain/colors/colormaps.html for available colormaps
                                         )
            else:
                axs[t_index, p_index].bar(projects.index(project) + 1, 0)
        axs[t_index, p_index].set_xlim([0, len(projects)+1])
        axs[t_index, p_index].set_ylim([0, 25])
        axs[t_index, p_index].spines["right"].set_visible(False)
        axs[t_index, p_index].spines["top"].set_visible(False)
        if p_index != 0:
            axs[t_index, p_index].spines["left"].set_visible(False)
            axs[t_index, p_index].tick_params(left = False, labelleft = False)
        else: #p_index==1, leftmost graph
            axs[t_index, p_index].set_ylabel('MdRE')
            axs[t_index, p_index].annotate(technique,
                                           xy=(0, 0.5), xytext=(-axs[t_index, p_index].yaxis.labelpad - 5, 0),
                                           xycoords=axs[t_index, p_index].yaxis.label, 
                                           textcoords='offset points',
                                           size=50, 
                                           ha='right', 
                                           va='center', rotation=90
                                          )
        if t_index == 0: # top graph
            axs[t_index, p_index].annotate(str(percentage) + "%", xy=(0.5, 1), 
                                           xytext=(0, 5),
                                           xycoords='axes fraction', 
                                           textcoords='offset points',
                                           size=50, 
                                           ha='center', 
                                           va='baseline')
        if t_index != len(techniques) - 1: #not bottom graph
            axs[t_index, p_index].spines["bottom"].set_visible(False)
            axs[t_index, p_index].tick_params(which='both',      # both major and minor ticks are affected
                                              bottom=False,      # ticks along the bottom edge are off
                                              top=False,         # ticks along the top edge are off
                                              labelbottom=False) # labels along the bottom edge are off
        else: #bottom graph
            axs[t_index, p_index].set_xticks(list(range(1, len(projects) + 1)))
            axs[t_index, p_index].set_xticklabels(projects, rotation=90, fontsize=24)
mpl.savefig("bargrid.eps")
import tikzplotlib
tikz_code = tikzplotlib.get_tikz_code(fig)

tikz_code = re.sub(r"SL", "\\\\SL", tikz_code)
tikz_code = re.sub(r"SC", "\\\\SC", tikz_code)
tikz_code = re.sub(r"SO", "\\\\SO", tikz_code)
tikz_code = re.sub(r"HG\\_([0-9])", "\\\\HG{\g<1>}", tikz_code)
tikz_code = re.sub(r"group\\_([0-9])", "\\\\OG{\g<1>}", tikz_code)
tikz_code = re.sub(r"\\_", "\\\\_", tikz_code)
tikz_code = re.sub(r"%", "\\\\%", tikz_code)
# print(tikz_code)

In [ ]:
full_df